In [1]:
from tvm.script import tir as T


@T.prim_func
def main(
    A: T.Buffer(128, "float32"),
    B: T.Buffer(128, "float32"),
    C: T.Buffer(128, "float32"),
):
    for i in range(128):
        with T.sblock("C"):
            vi = T.axis.spatial(128, i)
            C[vi] = A[vi] + B[vi]

[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[20:05:51] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: E

In [2]:
import IPython

IPython.display.Code(main.script())

# from tvm.script import tir as T

@T.prim_func
def main(A: T.Buffer((128,), "float32"), B: T.Buffer((128,), "float32"), C: T.Buffer((128,), "float32")):
    # with T.sblock("root"):
    for i in range(128):
        with T.sblock("C"):
            vi = T.axis.spatial(128, i)
            T.reads(A[vi], B[vi])
            T.writes(C[vi])
            C[vi] = A[vi] + B[vi]

In [3]:
import tvm

sch = tvm.s_tir.Schedule(main)
IPython.display.Code(sch.mod.script())

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(A: T.Buffer((128,), "float32"), B: T.Buffer((128,), "float32"), C: T.Buffer((128,), "float32")):
        # with T.sblock("root"):
        for i in range(128):
            with T.sblock("C"):
                vi = T.axis.spatial(128, i)
                T.reads(A[vi], B[vi])
                T.writes(C[vi])
                C[vi] = A[vi] + B[vi]

In [4]:
ex = tvm.compile(main, target="llvm")
# print(ex.mod.inspect_source())

In [5]:
import time

import numpy as np
from numpy.typing import NDArray


def lnumpy_mm_relu_ijk(
    A: NDArray[np.float64], B: NDArray[np.float64], C: NDArray[np.float64]
):
    Y = np.empty((128, 128), dtype="float64")
    for i in range(128):
        for j in range(128):
            for k in range(128):
                if k == 0:
                    Y[i, j] = 0
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
        for i in range(128):
            for j in range(128):
                C[i, j] = max(Y[i, j], 0)


def lnumpy_mm_relu_ikj(
    A: NDArray[np.float64], B: NDArray[np.float64], C: NDArray[np.float64]
):
    Y = np.empty((128, 128), dtype="float64")
    for i in range(128):
        for k in range(128):
            if A[i, k] == 0:
                continue
            for j in range(128):
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
        for i in range(128):
            for j in range(128):
                C[i, j] = max(Y[i, j], 0)


dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
c_np = np.empty((128, 128), dtype=dtype)

t0 = time.perf_counter()
lnumpy_mm_relu_ijk(a_np, b_np, c_np)
t1 = time.perf_counter()
lnumpy_mm_relu_ikj(a_np, b_np, c_np)
t2 = time.perf_counter()

print(f"np_mm_relu_ijk: {t1 - t0:.3f}s")
print(f"np_mm_reluikj: {t2 - t1:.3f}s")


np_mm_relu_ijk: 2.123s
np_mm_reluikj: 2.037s


In [6]:
dtype = "float32"


@tvm.script.ir_module
class MyModule:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((128, 128), dtype),
        B: T.Buffer((128, 128), dtype),
        C: T.Buffer((128, 128), dtype),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype)
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                vk = T.axis.reduce(128, k)
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


@tvm.script.ir_module
class MyModuleIKJ:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((128, 128), dtype),
        B: T.Buffer((128, 128), dtype),
        C: T.Buffer((128, 128), dtype),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype)
        for i, k, j in T.grid(128, 128, 128):
            with T.sblock("Y"):
                # vi = T.axis.spatial(128, i)
                # vj = T.axis.spatial(128, j)
                # vk = T.axis.reduce(128, k)
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                # vi = T.axis.spatial(128, i)
                # vj = T.axis.spatial(128, j)
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


In [7]:
n = 128

a_np = np.random.rand(n, n).astype("float32")
b_np = np.random.rand(n, n).astype("float32")
c_np = np.zeros((n, n), dtype="float32")

In [8]:
ex = tvm.compile(MyModule, target="llvm")
ev = ex.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev(a_np, b_np, c_np))


Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   2.0266       2.0266       2.0266       2.0266       0.0000                  


In [9]:
ex_ikj = tvm.compile(MyModuleIKJ, target="llvm")
ev_ijk = ex_ikj.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_ijk(a_np, b_np, c_np))


Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.2042       0.2042       0.2042       0.2042       0.0000                  


In [10]:
def lnumpy_mm_relu_v2(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((128, 128), dtype="float32")
    for i in range(128):
        for j0 in range(32):
            for k in range(128):
                for j1 in range(4):
                    j = j0 * 4 + j1
                    if k == 0:
                        Y[i, j] = 0
                    Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
    for i in range(128):
        for j in range(128):
            C[i, j] = max(Y[i, j], 0)


t0 = time.perf_counter()
lnumpy_mm_relu_v2(a_np, b_np, c_np)
t1 = time.perf_counter()

print(f"np_mm_relu_v2: {t1 - t0:.3f}s")

np_mm_relu_v2: 1.201s


In [11]:
import IPython

IPython.display.Code(MyModule.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(Y[vi, vj])
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [12]:
sch = tvm.s_tir.Schedule(MyModuleIKJ)

In [13]:
block_Y = sch.get_sblock("Y", func_name="mm_relu")
i, k, j = sch.get_loops(block_Y)

In [14]:
k0, k1 = sch.split(k, factors=[None, 4])

In [15]:
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, k_0, k_1, j in T.grid(128, 32, 4, 128):
            with T.sblock("Y"):
                vi, vj = T.axis.remap("SS", [i, j])
                vk = T.axis.reduce(128, k_0 * 4 + k_1)
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(Y[vi, vj])
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [16]:
sch.reorder(i, k0, j, k1)

In [17]:
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, k_0, j, k_1 in T.grid(128, 32, 128, 4):
            with T.sblock("Y"):
                vi, vj = T.axis.remap("SS", [i, j])
                vk = T.axis.reduce(128, k_0 * 4 + k_1)
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(Y[vi, vj])
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [18]:
ex_tiled = tvm.compile(sch.mod, target="llvm")
ev_tiled = ex_tiled.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_tiled(a_np, b_np, c_np))

Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.1488       0.1488       0.1488       0.1488       0.0000                  


In [19]:
block_C = sch.get_sblock("C", "mm_relu")
sch.reverse_compute_at(block_C, j)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, k_0, j in T.grid(128, 32, 128):
            for k_1 in range(4):
                with T.sblock("Y"):
                    vi, vj = T.axis.remap("SS", [i, j])
                    vk = T.axis.reduce(128, k_0 * 4 + k_1)
                    T.reads(A[vi, vk], B[vk, vj])
                    T.writes(Y[vi, vj])
                    with T.init():
                        Y[vi, vj] = T.float32(0.0)
                    Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [20]:
ex_fused = tvm.compile(sch.mod, target="llvm")
ev_fused = ex_fused.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_fused(a_np, b_np, c_np))

Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.1743       0.1743       0.1743       0.1743       0.0000                  


In [21]:
sch = tvm.s_tir.Schedule(MyModule)

block_Y = sch.get_sblock("Y", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)

j0, j1 = sch.split(j, factors=[None, 4])
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j_0, j_1, k in T.grid(128, 32, 4, 128):
            with T.sblock("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j_0 * 4 + j_1)
                vk = T.axis.reduce(128, k)
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(Y[vi, vj])
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [22]:
sch.reorder(j0, k, j1)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j_0, k, j_1 in T.grid(128, 32, 128, 4):
            with T.sblock("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j_0 * 4 + j_1)
                vk = T.axis.reduce(128, k)
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(Y[vi, vj])
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [23]:
ex_reordered = tvm.compile(sch.mod, target="llvm")
ev_reordered = ex_reordered.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_reordered(a_np, b_np, c_np))

Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.5190       0.5190       0.5190       0.5190       0.0000                  


In [24]:
block_C = sch.get_sblock("C", "mm_relu")
sch.reverse_compute_at(block_C, j0)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j_0 in T.grid(128, 32):
            for k, j_1 in T.grid(128, 4):
                with T.sblock("Y"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + j_1)
                    vk = T.axis.reduce(128, k)
                    T.reads(A[vi, vk], B[vk, vj])
                    T.writes(Y[vi, vj])
                    with T.init():
                        Y[vi, vj] = T.float32(0.0)
                    Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
            for ax0 in range(4):
                with T.sblock("C"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + ax0)
                    T.reads(Y[vi, vj])
                    T.writes(C[vi, vj])
                    C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [25]:
ex_fused = tvm.compile(sch.mod, target="llvm")
ev_fused = ex_fused.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_fused(a_np, b_np, c_np))

Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.5185       0.5185       0.5185       0.5185       0.0000                  


In [26]:
n = 1024


@tvm.script.ir_module
class MyModule1024:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((1024, 1024), "float32"),
        B: T.Buffer((1024, 1024), "float32"),
        C: T.Buffer((1024, 1024), "float32"),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((1024, 1024))
        for i, j, k in T.grid(1024, 1024, 1024):
            with T.sblock("Y"):
                vi = T.axis.spatial(1024, i)
                vj = T.axis.spatial(1024, j)
                vk = T.axis.reduce(1024, k)
                with T.init():
                    Y[vi, vj] = T.float32(0.0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(1024, 1024):
            with T.sblock("C"):
                vi = T.axis.spatial(1024, i)
                vj = T.axis.spatial(1024, j)
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))


a_np = np.random.rand(n, n).astype("float32")
b_np = np.random.rand(n, n).astype("float32")
c_np = np.zeros((n, n), dtype="float32")

# baseline ijk
ex_ijk = tvm.compile(MyModule1024, target="llvm")
ev_ijk = ex_ijk.mod.time_evaluator("mm_relu", tvm.cpu(), number=10)
print("ijk:", ev_ijk(a_np, b_np, c_np))

sch = tvm.s_tir.Schedule(MyModule1024)
block_Y = sch.get_sblock("Y", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)
j0, j1 = sch.split(j, factors=[None, 4])
sch.reorder(i, j0, k, j1)

block_C = sch.get_sblock("C", func_name="mm_relu")
sch.reverse_compute_at(block_C, j0)

ex_fused = tvm.compile(sch.mod, target="llvm")
ev_fused = ex_fused.mod.time_evaluator("mm_relu", tvm.cpu(), number=10)
print("fused:", ev_fused(a_np, b_np, c_np))


ijk: Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
 8711.5710    8711.5710    8711.5710    8711.5710      0.0000                  
fused: Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
 1932.5490    1932.5490    1932.5490    1932.5490      0.0000                  


In [27]:
sch = tvm.s_tir.Schedule(MyModule1024)

block_Y = sch.get_sblock("Y", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)
j0, j1 = sch.split(j, factors=[None, 4])
sch.reorder(i, j0, k, j1)

block_C = sch.get_sblock("C", func_name="mm_relu")
i_c, j_c = sch.get_loops(block_C)
sch.compute_at(block_Y, j_c)

In [28]:
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((1024, 1024), "float32"), B: T.Buffer((1024, 1024), "float32"), C: T.Buffer((1024, 1024), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((1024, 1024))
        for i, j in T.grid(1024, 1024):
            for ax0 in range(1024):
                with T.sblock("Y"):
                    vi, vj, vk = T.axis.remap("SSR", [i, j, ax0])
                    T.reads(A[vi, vk], B[vk, vj])
                    T.writes(Y[vi, vj])
                    with T.init():
                        Y[vi, vj] = T.float32(0.0)
                    Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                T.reads(Y[vi, vj])
                T.writes(C[vi, vj])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [ ]:
a_np = np.random.rand(n, n).astype("float32")
b_np = np.random.rand(n, n).astype("float32")
c_np = np.zeros((n, n), dtype="float32")


def bench(mod, label):
    ex = tvm.compile(mod, target="llvm")
    ev = ex.mod.time_evaluator("mm_relu", tvm.cpu(), number=10)
    ms = ev(a_np, b_np, c_np).mean * 1000
    print(f"{label}: {ms:.1f} ms")


# 1. baseline ijk
bench(MyModule1024, "ijk baseline")

# 2. compute_at(i) — Y row in L1/L2
sch = tvm.s_tir.Schedule(MyModule1024)
block_Y = sch.get_sblock("Y", func_name="mm_relu")
block_C = sch.get_sblock("C", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)
i_c, j_c = sch.get_loops(block_C)
sch.compute_at(block_Y, i_c)
bench(sch.mod, "compute_at(i) — Y row in L1/L2")

# 3. compute_at(j) — Y scalar in register, but ijk order
sch = tvm.s_tir.schedule(MyModule1024)
block_Y = sch.get_sblock("Y", func_name="mm_relu")
block_C = sch.get_sblock("C", func_name="mm_relu")
i_c, j_c = sch.get_loops(block_C)
sch.compute_at(block_Y, j_c)
bench(sch.mod, "compute_at(j) — Y scalar in reg, ijk order")

# 4. reverse_compute_at(j0) — Y tile in regs + cache friendly B access
sch = tvm.s_tir.Schedule(MyModule1024)
block_Y = sch.get_sblock("Y", func_name="mm_relu")
block_C = sch.get_sblock("C", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)
j0, j1 = sch.split(j, factors=[None, 4])
sch.reorder(i, j0, k, j1)
sch.reverse_compute_at(block_C, j0)
bench(sch.mod, "reverse_compute_at(j0) — Y tile in regs + ikj order")


ijk baseline: 8791.8 ms
compute_at(i) — Y row in L1/L2: 8438.1 ms


NameError: name 'mymodule1024' is not defined

In [ ]:
@tvm.script.ir_module
class MyModuleWithAxisRemapSugar:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((128, 128), "float32"),
        B: T.Buffer((128, 128), "float32"),
        C: T.Buffer((128, 128), "float32"),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype="float32")
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


sch = tvm.s_tir.Schedule(MyModuleWithAxisRemapSugar)

block_Y = sch.get_sblock("Y", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)

j0, j1 = sch.split(j, factors=[None, 4])

sch.reorder(j0, k, j1)

block_C = sch.get_sblock("C", "mm_relu")
sch.reverse_compute_at(block_C, j0)
IPython.display.Code(sch.mod.script(), language="python")


# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j_0 in T.grid(128, 32):
            for k, j_1 in T.grid(128, 4):
                with T.sblock("Y"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + j_1)
                    vk = T.axis.reduce(128, k)
                    T.reads(A[vi, vk], B[vk, vj])
                    T.writes(Y[vi, vj])
                    with T.init():
                        Y[vi, vj] = T.float32(0.0)
                    Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
            for ax0 in range(4):
                with T.sblock("C"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + ax0)
                    T.reads(Y[vi, vj])
                    T.writes(C[vi, vj])
                    C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))

In [ ]:
sch.decompose_reduction(block_Y, k)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tir.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((128, 128))
        for i, j_0 in T.grid(128, 32):
            for j_1_init in range(4):
                with T.sblock("Y_init"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + j_1_init)
                    T.reads()
                    T.writes(Y[vi, vj])
                    Y[vi, vj] = T.float32(0.0)
            for k, j_1 in T.grid(128, 4):
                with T.sblock("Y_update"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + j_1)
                    vk = T.axis.reduce(128, k)
                    T.reads(Y[vi, vj], A[vi, vk], B[vk, vj])
                    T.writes(Y[vi, vj])
                    Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
            for ax0 in range(4):
                with T.sblock("C"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + ax0)
                    T.reads(Y[vi, vj])
                    T.writes(C[vi, vj])
                    C[vi, vj] = T.max(Y[vi, vj], T.float32(0.0))